# Same Recipe, Real Scale: MNIST

You've already trained a CNN on scikit-learn's small 8x8 digit dataset (1,800 images). This notebook uses the *exact same idea* - conv, pool, conv, pool, flatten, decide - on the real thing: **MNIST**, 70,000 handwritten digits at 28x28 resolution.

What's actually new here, technically:
- A **real, standard dataset**, downloaded automatically (Colab's network handles this fine - this is exactly why we didn't rely on a classroom-wifi download earlier).
- A **DataLoader**, which handles batching and shuffling for us instead of the manual batching loop from before.
- Using the **GPU** if one's available (`Runtime -> Change runtime type -> GPU` if this notebook doesn't already have one selected).

Everything else - the layer types, the training loop shape - is the same recipe you already know.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## Load MNIST

`download=True` fetches the dataset the first time this runs. `transforms.ToTensor()` converts each image to a tensor and scales pixel values from 0-255 down to 0.0-1.0, same idea as the `/16.0` scaling from the small digit demo.


In [ ]:
transform = transforms.ToTensor()

train_data = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_data = torchvision.datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=256, shuffle=False)

print("Training examples:", len(train_data))
print("Test examples:", len(test_data))
image, label = train_data[0]
print("One image shape:", image.shape, "  label:", label)


## The network

> 🏷️ **B1-K1-W3** · Realiseert (onderdelen van) software — kwalificatiedossier Software development, kerntaak B1-K1

Same shape of network as before, just scaled for 28x28 input instead of 8x8. Trace the sizes through: 28x28 -> pool -> 14x14 -> pool -> 7x7, so the flattened size is `16 * 7 * 7 = 784`.

In [ ]:
class MnistCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(16 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = MnistCNN().to(device)
print(model)
print("\nTotal parameters:", sum(p.numel() for p in model.parameters()))


## Train

> 🏷️ **B1-K1-W3 · B1-K1-W4** · Realiseert (onderdelen van) software & Test software — kwalificatiedossier Software development, kerntaak B1-K1 (elke epoch traint én test de code meteen op de testset)

The DataLoader hands us one batch at a time, already shuffled - that's what the `for images, labels in train_loader:` line is doing. Everything inside the loop (forward pass, loss, zero_grad, backward, step) is exactly the training loop shape from before.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

EPOCHS = 3

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = loss_fn(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    avg_loss = running_loss / len(train_data)

    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            preds = model(images).argmax(dim=1)
            correct += (preds == labels).sum().item()
    accuracy = correct / len(test_data)

    print(f"epoch {epoch+1}/{EPOCHS}  |  loss {avg_loss:.4f}  |  test accuracy {accuracy:.1%}")


### What just changed, and what didn't

The dataset went from 1,800 tiny images to 70,000 real ones, and you're probably seeing 97-98%+ accuracy in just 3 epochs - better than the small demo, because there's simply more data to learn from. But notice what *didn't* change: the layer types, the training loop shape, even most of the code. That's the point - once you know the recipe, scaling up is mostly about data and infrastructure (DataLoader, GPU), not new ideas.

### Kwalificatie-koppeling
Deze notebook dekt B1-K1-W3 (Realiseert (onderdelen van) software) en B1-K1-W4 (Test software) uit kerntaak B1-K1 van het kwalificatiedossier mbo Software development (Crebo 23399, gewijzigd 2024). Volledige dekking over alle lessen heen: https://projectenplaats.nl/kwalificatiedossiers/software-development